# Phase 4 — Streaming & Controlled Drift Scenarios

Dự án: **An Evolving Fuzzy Reasoning System for Sensor Stream Anomaly Detection under Concept Drift**

Tập tin này độc lập với các notebook trước (Phase 1, 2, 3):
- Không sửa dữ liệu gốc.
- Mỗi kịch bản (scenario) được tạo từ một bản sao độc lập của Test stream (2.000 mẫu, UDI 8001 → 10000).
- Mục tiêu: Tạo giao thức thực nghiệm sạch, chuẩn hóa các kịch bản luồng để Static Fuzzy và Evolving Fuzzy được đánh giá đối đầu trên cùng một stream.
- Các giai đoạn:
  - **Phase 4.1**: Tạo Base Stream từ Test set
  - **Phase 4.2**: Định nghĩa kịch bản No-Drift
  - **Phase 4.3**: Thiết kế kịch bản Sudden Drift
  - **Phase 4.4**: Thiết kế kịch bản Gradual Drift
  - **Phase 4.5**: Kiểm tra và xác thực stream sau khi inject


In [1]:
# Phase 4.0 — Setup môi trường và nạp phân vùng Test stream
from pathlib import Path
import pandas as pd
import numpy as np

# Load dữ liệu gốc
data_path = Path("../data/ai4i2020.csv") if Path("../data/ai4i2020.csv").exists() else Path("data/ai4i2020.csv")
df = pd.read_csv(data_path)

sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# Trích xuất Test set (UDI 8001 -> 10000) theo Protocol Phase 1.5
test_df = df.iloc[8000:].copy()
print(f"Test set gốc nạp thành công: {len(test_df)} mẫu (UDI {test_df['UDI'].min()} → {test_df['UDI'].max()})")

# Universe of Discourse kế thừa từ Phase 2 (dùng cho clipping và mờ hóa)
universes = {
    "air_temp":     np.linspace(295.3, 304.5, 1000),
    "process_temp": np.linspace(305.7, 313.8, 1000),
    "rpm":          np.linspace(1168,  2886,  1000),
    "torque":       np.linspace(3.8,   76.2,  1000),
    "tool_wear":    np.linspace(0,     253,   1000),
    "anomaly":      np.linspace(0,     1,     1000),
}
print("Universes of Discourse đã được nạp sẵn sàng.")


Test set gốc nạp thành công: 2000 mẫu (UDI 8001 → 10000)
Universes of Discourse đã được nạp sẵn sàng.


In [2]:
# PHASE 4.1 — Create base sensor stream

stream_df = test_df.copy().reset_index(drop=True)

print("Base stream created")
print("-------------------")

print("Number of samples:", len(stream_df))

print("\nUDI range:")
print(
    stream_df["UDI"].iloc[0],
    "→",
    stream_df["UDI"].iloc[-1]
)

print("\nIndex range:")
print(
    stream_df.index[0],
    "→",
    stream_df.index[-1]
)

print("\nUDI is strictly increasing:",
      stream_df["UDI"].is_monotonic_increasing)

print("\nSensor columns:")
print(sensor_cols)

print("\nFirst 5 samples:")
print(
    stream_df[
        ["UDI"] + sensor_cols + ["Machine failure"]
    ].head()
)


Base stream created
-------------------
Number of samples: 2000

UDI range:
8001 → 10000

Index range:
0 → 1999

UDI is strictly increasing: True

Sensor columns:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

First 5 samples:
    UDI  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0  8001                300.8                    312.0                    1443   
1  8002                300.8                    312.0                    1374   
2  8003                300.9                    312.1                    1851   
3  8004                300.8                    312.0                    1274   
4  8005                300.9                    312.0                    1533   

   Torque [Nm]  Tool wear [min]  Machine failure  
0         53.3              151                0  
1         50.2              154                0  
2         23.0              156                0  
3         67.3   

## Phase 4.2 — Tạo Control Stream (No Injected Drift)

Tạo một bản sao độc lập của `stream_df` làm kịch bản kiểm chứng đối chứng:
- Đặt tên chuẩn hóa học thuật: **Control (No Injected Drift)** (thay vì No Drift đơn thuần, do tập Test gốc đã có sự dịch chuyển phân phối tự nhiên ghi nhận ở Phase 3).
- Chưa áp dụng bất kỳ tác động làm trôi dạt nhân tạo nào.
- Giữ nguyên vẹn toàn bộ 2.000 mẫu và phân bố nhãn lỗi (1.961 Normal, 39 Failure).


In [3]:
# PHASE 4.2 — Create control stream

control_stream = stream_df.copy()

print("Control stream created")
print("---------------------")

print("Number of samples:", len(control_stream))

print("\nUDI range:")
print(
    control_stream["UDI"].iloc[0],
    "→",
    control_stream["UDI"].iloc[-1]
)

print("\nData identical to base stream:",
      control_stream.equals(stream_df))

print("\nMachine failure distribution:")
print(
    control_stream["Machine failure"]
    .value_counts()
    .sort_index()
)

print("\nFirst 5 samples:")
print(
    control_stream[
        ["UDI"] + sensor_cols + ["Machine failure"]
    ].head()
)

Control stream created
---------------------
Number of samples: 2000

UDI range:
8001 → 10000

Data identical to base stream: True

Machine failure distribution:
Machine failure
0    1961
1      39
Name: count, dtype: int64

First 5 samples:
    UDI  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0  8001                300.8                    312.0                    1443   
1  8002                300.8                    312.0                    1374   
2  8003                300.9                    312.1                    1851   
3  8004                300.8                    312.0                    1274   
4  8005                300.9                    312.0                    1533   

   Torque [Nm]  Tool wear [min]  Machine failure  
0         53.3              151                0  
1         50.2              154                0  
2         23.0              156                0  
3         67.3              159                0  
4         44.0  

## Phase 4.3 — Thiết kế Kịch bản Sudden Drift

Mục tiêu: Thiết kế kịch bản trôi dạt đột ngột (Sudden Concept Drift) có kiểm soát trên nền Test stream:
- **Nguyên tắc bảo toàn Ground truth:** Giữ nguyên vẹn cột nhãn `Machine failure` để đảm bảo nền tảng đối chuẩn công bằng giữa Static Fuzzy và Evolving Fuzzy.
- **Lựa chọn kênh cảm biến ứng viên:** Tập trung vào 3 biến cơ học (`Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]`) - các kênh đã được kiểm định thống kê (KS-test ở Phase 3) cho thấy không bị ảnh hưởng bởi dịch chuyển tự nhiên.

### Phase 4.3.1 — Khảo sát phân phối 3 cảm biến ứng viên trên Test stream


In [4]:
# PHASE 4.3.1 — Inspect candidate drift sensors

candidate_drift_sensors = [
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

print("Candidate drift sensors")
print("=======================")

for col in candidate_drift_sensors:
    values = stream_df[col]

    print(f"\n{col}")
    print("-" * len(col))
    print(f"Mean   : {values.mean():.3f}")
    print(f"Std    : {values.std():.3f}")
    print(f"Min    : {values.min():.3f}")
    print(f"Q1     : {values.quantile(0.25):.3f}")
    print(f"Median : {values.median():.3f}")
    print(f"Q3     : {values.quantile(0.75):.3f}")
    print(f"Max    : {values.max():.3f}")

Candidate drift sensors

Rotational speed [rpm]
----------------------
Mean   : 1534.493
Std    : 165.329
Min    : 1181.000
Q1     : 1425.750
Median : 1504.000
Q3     : 1609.000
Max    : 2636.000

Torque [Nm]
-----------
Mean   : 40.025
Std    : 9.656
Min    : 12.100
Q1     : 33.300
Median : 40.100
Q3     : 46.400
Max    : 75.400

Tool wear [min]
---------------
Mean   : 106.246
Std    : 63.124
Min    : 0.000
Q1     : 51.000
Median : 106.000
Q3     : 161.000
Max    : 246.000


### Phase 4.3.2 — Tạo kịch bản Sudden Drift Stream

**Quyết định thiết kế:**
- **Kênh cảm biến tác động:** `Rotational speed [rpm]` và `Torque [Nm]`
- **Điểm trôi dạt (Drift Point):** `index = 1000` (phân chia đối xứng 50% trước - 50% sau: `0 → 999` và `1000 → 1999`).
- **Mức độ dịch chuyển (Shift):**
  - RPM shift: **−150 rpm** ($pprox 0.91\sigma_{	ext{test}}$)
  - Torque shift: **+8 Nm** ($pprox 0.83\sigma_{	ext{test}}$)
- **Bảo toàn nhãn:** Giữ nguyên vẹn $100\%$ nhãn `Machine failure` ($P(X)$ feature drift, giữ nguyên ground truth).
- **Kiểm soát biên:** Clipping theo miền vũ luận của FIS (`RPM: [1168, 2886]`, `Torque: [3.8, 76.2]`).


In [5]:
# PHASE 4.3.2 — Create sudden drift stream

import numpy as np

SUDDEN_DRIFT_POINT = 1000

RPM_SHIFT = -150
TORQUE_SHIFT = 8

RPM_MIN = universes["rpm"].min()
RPM_MAX = universes["rpm"].max()

TORQUE_MIN = universes["torque"].min()
TORQUE_MAX = universes["torque"].max()

sudden_drift_stream = stream_df.copy()

# Apply sudden drift after the drift point
after_drift = sudden_drift_stream.index >= SUDDEN_DRIFT_POINT

sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] = (
    sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"]
    + RPM_SHIFT
).clip(RPM_MIN, RPM_MAX)

sudden_drift_stream.loc[after_drift, "Torque [Nm]"] = (
    sudden_drift_stream.loc[after_drift, "Torque [Nm]"]
    + TORQUE_SHIFT
).clip(TORQUE_MIN, TORQUE_MAX)

print("Sudden drift stream created")
print("==========================")

print("Drift point:", SUDDEN_DRIFT_POINT)

print("Before drift:",
      f"0 → {SUDDEN_DRIFT_POINT - 1}")

print("After drift:",
      f"{SUDDEN_DRIFT_POINT} → {len(sudden_drift_stream) - 1}")

print("\nRPM shift:", RPM_SHIFT)
print("Torque shift:", TORQUE_SHIFT)

print("\nRPM range after injection:")
print(
    sudden_drift_stream["Rotational speed [rpm]"].min(),
    "→",
    sudden_drift_stream["Rotational speed [rpm]"].max()
)

print("\nTorque range after injection:")
print(
    sudden_drift_stream["Torque [Nm]"].min(),
    "→",
    sudden_drift_stream["Torque [Nm]"].max()
)

print("\nMachine failure distribution:")
print(
    sudden_drift_stream["Machine failure"]
    .value_counts()
    .sort_index()
)

print("\nUDI unchanged:",
      sudden_drift_stream["UDI"].equals(stream_df["UDI"]))

print("\nLabels unchanged:",
      sudden_drift_stream["Machine failure"].equals(
          stream_df["Machine failure"]
      ))

Sudden drift stream created
Drift point: 1000
Before drift: 0 → 999
After drift: 1000 → 1999

RPM shift: -150
Torque shift: 8

RPM range after injection:
1168 → 2617

Torque range after injection:
12.1 → 76.2

Machine failure distribution:
Machine failure
0    1961
1      39
Name: count, dtype: int64

UDI unchanged: True

Labels unchanged: True


### Phase 4.3.3 — Xác thực thống kê kịch bản Sudden Drift

Kiểm tra định lượng sự thay đổi phân phối trước và sau điểm trôi dạt (`index = 1000`):
- So sánh Mean và Median của 3 biến cơ học (`Rotational speed`, `Torque`, `Tool wear`).
- Kiểm định Kolmogorov-Smirnov 2 mẫu (Two-Sample KS Test) để đo lường khoảng cách phân bố ($D$) và mức ý nghĩa thống kê ($p$-value).
- *Lưu ý diễn giải học thuật:* Phép kiểm định KS xác nhận sự khác biệt phân phối cảm biến thực nghiệm qua ranh giới trôi dạt, không ngộ nhận là bằng chứng độc lập của quan hệ nhân quả lỗi.


In [6]:
# PHASE 4.3.3 — Validate sudden drift

from scipy.stats import ks_2samp

before = sudden_drift_stream.iloc[:SUDDEN_DRIFT_POINT]
after = sudden_drift_stream.iloc[SUDDEN_DRIFT_POINT:]

print("Sudden Drift Validation")
print("=======================")

for col in candidate_drift_sensors:
    before_values = before[col]
    after_values = after[col]

    ks_stat, p_value = ks_2samp(
        before_values,
        after_values
    )

    print(f"\n{col}")
    print("-" * len(col))

    print(f"Before mean   : {before_values.mean():.3f}")
    print(f"After mean    : {after_values.mean():.3f}")

    print(f"Before median : {before_values.median():.3f}")
    print(f"After median  : {after_values.median():.3f}")

    print(f"Mean shift    : "
          f"{after_values.mean() - before_values.mean():.3f}")

    print(f"KS statistic  : {ks_stat:.4f}")
    print(f"KS p-value    : {p_value:.6g}")

Sudden Drift Validation

Rotational speed [rpm]
----------------------
Before mean   : 1528.022
After mean    : 1391.699
Before median : 1501.000
After median  : 1357.000
Mean shift    : -136.323
KS statistic  : 0.4300
KS p-value    : 2.32384e-83

Torque [Nm]
-----------
Before mean   : 40.253
After mean    : 47.789
Before median : 40.050
After median  : 48.100
Mean shift    : 7.536
KS statistic  : 0.3080
KS p-value    : 2.7946e-42

Tool wear [min]
---------------
Before mean   : 105.255
After mean    : 107.237
Before median : 104.500
After median  : 108.000
Mean shift    : 1.982
KS statistic  : 0.0240
KS p-value    : 0.93577


## Phase 4.4 — Thiết kế Kịch bản Gradual Drift

Mục tiêu: Thiết kế kịch bản trôi dạt tiệm tiến (Gradual Concept Drift) với các thông số đồng nhất về cường độ cực đại với Sudden Drift:
- **Cường độ trôi dạt cuối cùng:** RPM shift **−150 rpm**, Torque shift **+8 Nm**.
- **Cửa sổ chuyển tiếp (Transition Window):** 400 mẫu, từ `index = 800` đến `1199` (tâm chuyển tiếp tại `index = 1000`, tương ứng $\alpha = 0.5$).
  - `0 → 799`: Trạng thái ổn định trước trôi dạt ($\alpha = 0.0$).
  - `800 → 1199`: Vùng chuyển tiếp trôi dạt tuyến tính ($\alpha$ tăng đều từ $0.0 \to 1.0$).
  - `1200 → 1999`: Trạng thái ổn định sau trôi dạt hoàn toàn ($\alpha = 1.0$).
- **Bảo toàn dữ liệu:** Giữ nguyên 100% nhãn `Machine failure` và thứ tự `UDI`. Kiểm soát biên an toàn bằng clipping theo universe của FIS.

### Phase 4.4.1 — Tạo Gradual Drift Stream


In [7]:
# PHASE 4.4.1 — Create gradual drift stream

GRADUAL_DRIFT_START = 800
GRADUAL_DRIFT_END = 1200

gradual_drift_stream = stream_df.copy()

for i in gradual_drift_stream.index:
    if i < GRADUAL_DRIFT_START:
        alpha = 0.0

    elif i >= GRADUAL_DRIFT_END:
        alpha = 1.0

    else:
        alpha = (
            (i - GRADUAL_DRIFT_START)
            / (GRADUAL_DRIFT_END - GRADUAL_DRIFT_START)
        )

    gradual_drift_stream.loc[
        i, "Rotational speed [rpm]"
    ] = np.clip(
        stream_df.loc[i, "Rotational speed [rpm]"]
        + alpha * RPM_SHIFT,
        RPM_MIN,
        RPM_MAX
    )

    gradual_drift_stream.loc[
        i, "Torque [Nm]"
    ] = np.clip(
        stream_df.loc[i, "Torque [Nm]"]
        + alpha * TORQUE_SHIFT,
        TORQUE_MIN,
        TORQUE_MAX
    )

print("Gradual drift stream created")
print("===========================")

print("Transition start:", GRADUAL_DRIFT_START)
print("Transition end  :", GRADUAL_DRIFT_END)

print("\nBefore transition:",
      f"0 → {GRADUAL_DRIFT_START - 1}")

print("Transition window:",
      f"{GRADUAL_DRIFT_START} → {GRADUAL_DRIFT_END - 1}")

print("After transition:",
      f"{GRADUAL_DRIFT_END} → {len(gradual_drift_stream) - 1}")

print("\nFinal RPM shift:", RPM_SHIFT)
print("Final Torque shift:", TORQUE_SHIFT)

print("\nMachine failure distribution:")
print(
    gradual_drift_stream["Machine failure"]
    .value_counts()
    .sort_index()
)

print("\nUDI unchanged:",
      gradual_drift_stream["UDI"].equals(
          stream_df["UDI"]
      ))

print("\nLabels unchanged:",
      gradual_drift_stream["Machine failure"].equals(
          stream_df["Machine failure"]
      ))

C:\Users\vinhv\AppData\Local\Temp\ipykernel_59500\1549547564.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1487.625' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gradual_drift_stream.loc[


Gradual drift stream created
Transition start: 800
Transition end  : 1200

Before transition: 0 → 799
Transition window: 800 → 1199
After transition: 1200 → 1999

Final RPM shift: -150
Final Torque shift: 8

Machine failure distribution:
Machine failure
0    1961
1      39
Name: count, dtype: int64

UDI unchanged: True

Labels unchanged: True


### Phase 4.4.2 — Xác thực thống kê kịch bản Gradual Drift

Kiểm tra định lượng sự thay đổi phân phối giữa hai phân đoạn ổn định:
- **Trước chuyển tiếp (Before):** `0 → 799` (800 mẫu ban đầu).
- **Sau chuyển tiếp (After):** `1200 → 1999` (800 mẫu sau khi trôi dạt hoàn toàn).
- *Lưu ý thiết kế:* Cửa sổ chuyển tiếp `800 → 1199` (400 mẫu) được tách riêng, so sánh trực tiếp hai trạng thái ổn định trước và sau drift.


In [8]:
# PHASE 4.4.2 — Validate gradual drift

from scipy.stats import ks_2samp

before = gradual_drift_stream.iloc[:GRADUAL_DRIFT_START]
after = gradual_drift_stream.iloc[GRADUAL_DRIFT_END:]

print("Gradual Drift Validation")
print("========================")

print("Before transition:",
      f"0 → {GRADUAL_DRIFT_START - 1}")

print("After transition:",
      f"{GRADUAL_DRIFT_END} → {len(gradual_drift_stream) - 1}")

for col in candidate_drift_sensors:
    before_values = before[col]
    after_values = after[col]

    ks_stat, p_value = ks_2samp(
        before_values,
        after_values
    )

    print(f"\n{col}")
    print("-" * len(col))

    print(f"Before mean   : {before_values.mean():.3f}")
    print(f"After mean    : {after_values.mean():.3f}")

    print(f"Before median : {before_values.median():.3f}")
    print(f"After median  : {after_values.median():.3f}")

    print(f"Mean shift    : "
          f"{after_values.mean() - before_values.mean():.3f}")

    print(f"KS statistic  : {ks_stat:.4f}")
    print(f"KS p-value    : {p_value:.6g}")

Gradual Drift Validation
Before transition: 0 → 799
After transition: 1200 → 1999

Rotational speed [rpm]
----------------------
Before mean   : 1527.561
After mean    : 1388.809
Before median : 1500.000
After median  : 1353.000
Mean shift    : -138.753
KS statistic  : 0.4288
KS p-value    : 2.29379e-66

Torque [Nm]
-----------
Before mean   : 40.174
After mean    : 47.972
Before median : 40.000
After median  : 48.350
Mean shift    : 7.797
KS statistic  : 0.3212
KS p-value    : 6.68471e-37

Tool wear [min]
---------------
Before mean   : 105.154
After mean    : 108.981
Before median : 104.000
After median  : 109.500
Mean shift    : 3.828
KS statistic  : 0.0350
KS p-value    : 0.711572


## Phase 4.5 — Kiểm tra Toàn diện Tính Nhất quán và Toàn vẹn của 3 Kịch bản Stream

Mục tiêu: Đảm bảo cả 3 kịch bản thử nghiệm (**Control**, **Sudden Drift**, **Gradual Drift**) tuân thủ tuyệt đối chuẩn mực đối chuẩn công bằng:
- Cùng kích thước (2.000 mẫu) và cùng dải `UDI` (`8001 → 10000`).
- Giữ nguyên vẹn 100% thứ tự luồng đơn điệu và nhãn mục tiêu `Machine failure` (39 ca lỗi).
- Bảo đảm tính nhất quán nhãn chéo (cross-scenario label consistency).

### Phase 4.5.1 — Integrity check


In [9]:
# PHASE 4.5.1 — Validate all experimental streams

scenario_streams = {
    "Control": control_stream,
    "Sudden Drift": sudden_drift_stream,
    "Gradual Drift": gradual_drift_stream,
}

print("Experimental Stream Integrity Check")
print("===================================")

for name, df in scenario_streams.items():

    print(f"\n{name}")
    print("-" * len(name))

    print("Samples:", len(df))

    print(
        "UDI range:",
        df["UDI"].iloc[0],
        "→",
        df["UDI"].iloc[-1]
    )

    print(
        "UDI unchanged:",
        df["UDI"].equals(stream_df["UDI"])
    )

    print(
        "Labels unchanged:",
        df["Machine failure"].equals(
            stream_df["Machine failure"]
        )
    )

    print(
        "Failure count:",
        df["Machine failure"].sum()
    )

print("\nCross-scenario label consistency:")
print(
    control_stream["Machine failure"].equals(
        sudden_drift_stream["Machine failure"]
    )
    and
    control_stream["Machine failure"].equals(
        gradual_drift_stream["Machine failure"]
    )
)

Experimental Stream Integrity Check

Control
-------
Samples: 2000
UDI range: 8001 → 10000
UDI unchanged: True
Labels unchanged: True
Failure count: 39

Sudden Drift
------------
Samples: 2000
UDI range: 8001 → 10000
UDI unchanged: True
Labels unchanged: True
Failure count: 39

Gradual Drift
-------------
Samples: 2000
UDI range: 8001 → 10000
UDI unchanged: True
Labels unchanged: True
Failure count: 39

Cross-scenario label consistency:
True


### Phase 4.5.2 — Kiểm tra biệt hóa cảm biến cuối cùng (Final Scenario Difference Check)

Xác minh tính độc lập và độ chính xác của quá trình can thiệp (perturbation):
- **Kỳ vọng Control:** Toàn bộ 5 cảm biến không đổi (UNCHANGED).
- **Kỳ vọng Sudden Drift:** Chỉ duy nhất 2 cảm biến cơ học (`Rotational speed` và `Torque`) thay đổi (CHANGED); 3 cảm biến còn lại không đổi (UNCHANGED).
- **Kỳ vọng Gradual Drift:** Chỉ duy nhất 2 cảm biến cơ học (`Rotational speed` và `Torque`) thay đổi (CHANGED); 3 cảm biến còn lại không đổi (UNCHANGED).


In [10]:
# PHASE 4.5.2 — Final scenario difference check

comparison_sensors = sensor_cols

print("Final Scenario Difference Check")
print("==============================")

for name, df in [
    ("Control", control_stream),
    ("Sudden Drift", sudden_drift_stream),
    ("Gradual Drift", gradual_drift_stream),
]:
    print(f"\n{name}")
    print("-" * len(name))

    for col in comparison_sensors:
        changed = not df[col].equals(stream_df[col])

        print(
            f"{col}:",
            "CHANGED" if changed else "UNCHANGED"
        )

print("\nExpected:")
print("- Control: all sensors UNCHANGED")
print("- Sudden Drift: only RPM and Torque CHANGED")
print("- Gradual Drift: only RPM and Torque CHANGED")

Final Scenario Difference Check

Control
-------
Air temperature [K]: UNCHANGED
Process temperature [K]: UNCHANGED
Rotational speed [rpm]: UNCHANGED
Torque [Nm]: UNCHANGED
Tool wear [min]: UNCHANGED

Sudden Drift
------------
Air temperature [K]: UNCHANGED
Process temperature [K]: UNCHANGED
Rotational speed [rpm]: CHANGED
Torque [Nm]: CHANGED
Tool wear [min]: UNCHANGED

Gradual Drift
-------------
Air temperature [K]: UNCHANGED
Process temperature [K]: UNCHANGED
Rotational speed [rpm]: CHANGED
Torque [Nm]: CHANGED
Tool wear [min]: UNCHANGED

Expected:
- Control: all sensors UNCHANGED
- Sudden Drift: only RPM and Torque CHANGED
- Gradual Drift: only RPM and Torque CHANGED
